# Multilayer CNN
## On MNIST dataset

In [30]:
import tensorflow_datasets as tfds
import tensorflow as tf

In [31]:
# loading the data
mnist_bldr = tfds.builder('mnist')
mnist_bldr.download_and_prepare()
# shuffel false, beacuse we will split the training set into two parts: smaller training dataset and a validation dataset
# if shuffling was on, it would incur resuffling of the dataset each time we fetched a mini-batch of data
# can cause false performance estimation of the model
datasets = mnist_bldr.as_dataset(shuffle_files=False)
mnist_train_orig = datasets['train']
mnist_test_orig = datasets['test']

In [32]:
# preparing the dataset
BUFFER_SIZE = 10000
BATCH_SIZE = 64
NUM_EPOCHS = 20

mnist_train = mnist_train_orig.map(
    lambda item: (tf.cast(item['image'], tf.float32) / 255.0,
                  tf.cast(item['label'], tf.int32))
)

mnist_test = mnist_test_orig.map(
    lambda item: (tf.cast(item['image'], tf.float32) / 255.0,
                  tf.cast(item['label'], tf.int32))
)

tf.random.set_seed(1)

mnist_train = mnist_train.shuffle(buffer_size=BUFFER_SIZE,
                                  reshuffle_each_iteration = False)


mnist_valid = mnist_train.take(10000).batch(BATCH_SIZE)
mnist_train = mnist_train.skip(10000).batch(BATCH_SIZE)

In [33]:
# constructing the cnn
model = tf.keras.Sequential()
model.add(tf.keras.layers.Conv2D(
    filters=32, kernel_size=(5,5),
    strides=(1,1), padding='same',
    data_format='channels_last',
    name='conv_1', activation='relu')
)

model.add(tf.keras.layers.MaxPool2D(
    pool_size=(2,2), name='pool_1'
))

model.add(tf.keras.layers.Conv2D(
    filters=64, kernel_size=(5,5),
    strides=(1,1), padding='same',
    name='conv_2', activation='relu'
))

model.add(tf.keras.layers.MaxPool2D(
    pool_size=(2,2), name='pool_2'
))

In [34]:
# first dimension is batch dimension, used 16 for abitray,
# could have used none also
model.compute_output_shape(input_shape=(16,28,28,1))

(16, 7, 7, 64)

In [35]:
# next layer we want to add is dense (fully conected) 
# for implementing a classifier on top of conventionoal and pool layers
# input to this layer must have rank 2, that is the shape [batch_size x input_units]
# thus we need to flatten the output from previous layers to meet the requirments for the dense layer

model.add(tf.keras.layers.Flatten())
# verify that it has been faltened
model.compute_output_shape(input_shape=(16, 28, 28, 1))

(16, 3136)

In [36]:
# add two dense layers, with dropout in-between
model.add(tf.keras.layers.Dense(
    units=1024, name='fc1',
    activation='relu'
))

# half of the neurons are set to 0 (ps: only during training)
model.add(tf.keras.layers.Dropout(
    rate=0.5
))

model.add(tf.keras.layers.Dense(
    units=10, name='fc_2',
    activation='softmax'
))

In [ ]:
# uses SpareseCategoricalCrossentopy because we have integer (sparse) labels instead of one-hot-encoded
tf.random.set_seed(1)
model.build(input_shape=(None, 28, 28, 1))

# adam optimizer, is a robust gradient-desect based optimization
# suited for nonconvex optimization and machine learning
# key advantage of adam is the choice of update step size derived from the running averrage of gradient movments
model.compile(
    optimizer = tf.keras.optimizers.Adam(),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = ['accuracy']
)

model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_1 (Conv2D)                 │ (None, 28, 28, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling2D)           │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv2D)                 │ (None, 14, 14, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling2D)           │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 1024)           │     3,212,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_2 (Dense)                    │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,274,634 (12.49 MB)

 Trainable params: 3,274,634 (12.49 MB)

 Non-trainable params: 0 (0.00 B)